# Histogram selector with local scale refinement

Choose a 4-bit format that keeps the rounded multiplication close to `W @ X_cal`. The selector uses a histogram estimate, local scale improvements, and checks on sampled weight rows.

The selector does not exhaustively search every combination of block exponents. A separate exhaustive reference is included later only to evaluate its choice. Set the data paths and options below.



In [1]:
import os, time, math, itertools
import numpy as np, pandas as pd

W_PATH      = "c:/Users/Admin/OneDrive/Desktop/System/W_C.npy"
X_CAL_PATH  = "c:/Users/Admin/OneDrive/Desktop/System/X_cal.npy"
X_TEST_PATH = None

RUN_REFERENCE_CHECK = True
RUN_TEST_SUITE      = False

GHS_PARAMS = dict(hw = 3, n_tail = 15, head_frac = 0.5, head_cap = 0.25, z = 3.0, early_margin = 1.3, early_coh = 0.10, ncls = 8, keep_hist_leader = True)


## 1. Number formats

The candidates are E2M1, E1M2, MXINT4, MXUINT4, and BFP4. Each format allows a small set of rounded values. Blocks of up to 32 weights within a row share a scale, which sets the spacing between allowed values.

MXUINT4 uses stored codes 0–15 decoded by subtracting 8, so this custom convention can reconstruct negative values too. All stages use the same format settings.



In [2]:
E8M0_MIN_EXP, E8M0_MAX_EXP = - 127, 127
BFP_EXP_POLICY = "autoflex_predict"
BFP_ALPHA = 1.0

BFP_SHARING = "block"
BFP_BLOCK_SIZE = 32

FORMAT_CONFIGS = {
    "E2M1": {
        "kind": "mxfp",
        "block_size": 32,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "[S|E|M] = [1|2|1]",
        "grid": np.array(
            [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0],
            dtype = np.float64
        ),
        "baseline_exp_shift": - 2,
        "exp_min": E8M0_MIN_EXP,
        "exp_max": E8M0_MAX_EXP,
        "shared_scale": "power-of-two 2^e; 8-bit E8M0-style shared exponent",
        "exceptions": "finite values only; no NaN/Inf codes are emitted",
        "convention_status": "classroom/OCP-MX convention used in this project",
    },

    "E1M2": {
        "kind": "mxfp",
        "block_size": 32,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "[S|E|M] = [1|1|2]",
        "grid": np.array(
            [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5],
            dtype = np.float64
        ),
        "baseline_exp_shift": - 1,
        "exp_min": E8M0_MIN_EXP,
        "exp_max": E8M0_MAX_EXP,
        "shared_scale": "power-of-two 2^e; 8-bit E8M0-style shared exponent",
        "exceptions": "finite values only; no NaN/Inf codes are emitted",
        "convention_status": "adopted classroom-consistent E1M2 convention",
    },

    "MXINT4": {
        "kind": "mxint",
        "block_size": 32,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "[S|I|F] = [1|1|2]",
        "qmin": - 8,
        "qmax": 7,
        "element_scale": 0.25,
        "exp_min": E8M0_MIN_EXP,
        "exp_max": E8M0_MAX_EXP,
        "shared_scale": "power-of-two 2^e; element precision scale 2^-2",
        "exceptions": "signed integer codes -8..7; no NaN/Inf codes",
        "convention_status": "adopted from the classroom MXINT construction",
    },

    "BFP4": {
        "kind": "bfp",
        "block_size": BFP_BLOCK_SIZE,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "flex4+8: 4-bit two's-complement integer mantissa (codes -8..7) + shared exponent",
        "mantissa_bits": 4,
        "qmin": - 8,
        "qmax": 7,
        "exp_min": E8M0_MIN_EXP,
        "exp_max": E8M0_MAX_EXP,
        "shared_scale": "kappa = 2^E, value = m * kappa; 8-bit shared exponent per group (paper: M-bit exponent shared across the tensor)",
        "exceptions": "finite two's-complement integers only; saturate to [-8,7]; no NaN/Inf codes",
        "convention_status": "Flexpoint (Koster et al., arXiv:1711.02213): N-bit two's-complement mantissa, shared exponent, Autoflex exponent rule; group size / exponent width / rounding are course-level choices (see BFP_* switches)",
    },

    "MXUINT4": {
        "kind": "mxuint",
        "block_size": 32,
        "element_bits": 4,
        "metadata_bits": 8,
        "layout": "Unsigned 4-bit code; fixed midpoint zero point q_u - 8",
        "shared_scale": "power-of-two 2^e; fixed zero point 8; no per-block offset",
        "exceptions": "finite values; unsigned codes 0..15 decode to signed integers -8..7",
        "convention_status": "custom midpoint-coded unsigned-storage convention; no FP32 offset overhead",
    },
}


## 2. Rounding and error helpers

The next cell defines block boundaries, baseline exponents, rounding, and full output error. An exponent sets a power-of-two scale. Values round to allowed levels with ties-to-even and clip at the range limits.

The Gram factor gives an equivalent way to calculate the same output-error objective with fewer input dimensions. It does not change what error is being measured.



In [3]:
import math, numpy as np
BLOCK = 32

EXP_MIN, EXP_MAX = - 127, 127

FORMATS = ["E2M1", "E1M2", "MXUINT4", "MXINT4", "BFP4"]

GRID = {"E2M1": np.array([0, .5, 1, 1.5, 2, 3, 4, 6.]), "E1M2": np.array([0, .5, 1, 1.5, 2, 2.5, 3, 3.5])}

FP_SHIFT = {"E2M1": - 2, "E1M2": - 1}

KIND = {"E2M1": "mxfp", "E1M2": "mxfp", "MXINT4": "mxint", "BFP4": "bfp", "MXUINT4": "mxuint"}

BFP_N = 4

def block_ranges(cols, bs = BLOCK):
    return [(s, min(s + bs, cols)) for s in range(0, cols, bs)]

def baseline_exps(Wd, fmt):
    """(rows, nb) baseline exponent per block, same rule as notebook baseline_exp()."""
    rows, cols = Wd.shape
    out = np.zeros((rows, len(block_ranges(cols))), dtype = np.int64)
    for b, (s, e) in enumerate(block_ranges(cols)):
        blk = Wd[:, s:e]
        amax = np.max(np.abs(blk), axis = 1)
        nz = amax > 0
        with np.errstate(divide = "ignore"):
            lg = np.where(nz, np.log2(np.where(nz, amax, 1.0)), 0.0)
        k = KIND[fmt]
        if k == "mxfp":
            ex = np.floor(lg).astype(np.int64) + FP_SHIFT[fmt]
        elif k == "mxint":
            ex = np.floor(lg).astype(np.int64)
        elif k == "bfp":
            ex = np.ceil(lg).astype(np.int64) - BFP_N + 1
        else:
            neg = np.maximum(0.0, - np.min(blk, axis = 1)); pos = np.maximum(0.0, np.max(blk, axis = 1))
            req = np.maximum(neg / 8.0, pos / 7.0)
            ok = req > 0
            with np.errstate(divide = "ignore"):
                ex = np.ceil(np.log2(np.where(ok, req, 1.0))).astype(np.int64)
            ex = np.where(ok, ex, 0)
        ex = np.where(nz, ex, 0)
        out[:, b] = np.clip(ex, EXP_MIN, EXP_MAX)
    return out

def _fp_round(local, grid):
    """round non-negative `local` to grid, ties to even index (matches nearest_grid_rne)."""
    mid = 0.5 * (grid[: - 1] + grid[1:])
    idx = np.searchsorted(mid, local, side = "left")
    i = np.minimum(idx, len(mid) - 1)
    tie = (idx < len(mid)) & (np.abs(local - mid[i]) <= 1e-15)

    j = np.clip(idx - 1, 0, len(mid) - 1)
    tie_hi = (idx > 0) & (np.abs(local - mid[j]) <= 1e-15) & ~tie
    lo = np.where(tie, i, np.where(tie_hi, j, idx))
    pick = np.where((tie | tie_hi), np.where(lo % 2 == 0, lo, lo + 1), idx)
    return grid[np.minimum(pick, len(grid) - 1)]

def quantize(x, fmt, exp):
    """x: float64 array, exp: int array broadcastable to x. Returns dequantised values."""
    exp = np.clip(exp, EXP_MIN, EXP_MAX)
    scale = np.exp2(exp.astype(np.float64))
    k = KIND[fmt]
    if k == "mxfp":
        q = _fp_round(np.abs(x) / scale, GRID[fmt])
        return np.sign(x) * q * scale
    if k == "mxint":
        step = scale * 0.25
        return np.clip(np.rint(x / step), - 8, 7) * step
    return np.clip(np.rint(x / scale), - 8, 7) * scale

def gram_factor(X):
    G = X @ X.T
    lam, V = np.linalg.eigh(G)
    if lam.size == 0 or np.max(lam) <= 0:
        return np.zeros((X.shape[0], 0))
    keep = lam > np.max(lam) * 1e-14
    return V[:, keep] * np.sqrt(lam[keep])

def err_pct(Wd, Wq, X):
    den = np.linalg.norm(Wd @ X)
    return 0.0 if den == 0 else 100.0 * np.linalg.norm((Wd - Wq) @ X) / den


## 3. Estimate format errors using histograms

1. Normalize each weight magnitude by its block's baseline unit.
2. Group similar blocks into classes and put their normalized values into small bins. Keep signs separate when positive and negative ranges differ.
3. Weight the summaries by input-channel energy, so active inputs contribute more to the estimate.
4. For each class, try scale offsets using bin summaries and keep the lowest estimated squared error. Add these estimates to rank formats. The chosen offsets also give starting exponents for individual blocks.

**Example:** a bin contains values 1.1 and 1.2 with equal unit weights. If a candidate rounds them to 1, squared error is `0.1² + 0.2² = 0.05`. Using their count 2, sum 2.3, and squared sum 2.65 gives the same result: `2.65 − 2 × 1 × 2.3 + 1² × 2 = 0.05`. This illustrates why summaries can avoid revisiting each value for every candidate. Actual scoring also includes scales and input energies.

The histogram estimate does not capture errors cancelling or reinforcing across input channels. It helps screen formats, but does not guarantee the best full-output choice.



In [4]:
NBINS_PER_UNIT = 128

NB_T = 8 * NBINS_PER_UNIT

OFFSETS = tuple(range( - 4, 4))

def _code_round(tp, fmt, neg):
    if KIND[fmt] == "mxfp":
        return _fp_round(tp, GRID[fmt])
    hi = 8.0 if neg else 7.0
    return np.clip(np.rint(tp), 0, hi)

def block_classes_fmt(t, bidx, rows, nbk, ncls, Wd, mode = "tmax"):
    """Class of every block from where its largest normalised magnitude sits inside its octave (decides which
    exponent offset is best for that block), optionally split again by peakedness (rms/amax)."""
    tm = np.zeros((rows, nbk)); rm = np.zeros((rows, nbk))
    for b, (s_, e_) in enumerate(block_ranges(Wd.shape[1])):
        tm[:, b] = t[:, s_:e_].max(axis = 1)
        rm[:, b] = np.sqrt((t[:, s_:e_] ** 2).mean(axis = 1)) / np.where(tm[:, b] > 0, tm[:, b], 1)
    if ncls <= 1: return np.zeros((rows, nbk), dtype = np.int64)
    a = max(2, int(round(ncls ** 0.5))) if MODE2D else ncls
    lt = np.log2(np.where(tm > 0, tm, 1.0)); fr = lt - np.floor(lt)
    c1 = np.minimum((fr * a).astype(np.int64), a - 1)
    if not MODE2D: return c1
    b2 = max(1, ncls // a); edges = np.quantile(rm, np.linspace(0, 1, b2 + 1)[1: - 1]) if b2 > 1 else []
    c2 = np.searchsorted(edges, rm, side = "right") if b2 > 1 else 0
    return c1 * b2 + c2

MODE2D = False

# Estimate error from weighted histogram summaries.
def surrogate(Wd, g, fmt, counters = None, ncls = 8, classes = None):
    """Estimate input-weighted error using class histograms."""
    rows, cols = Wd.shape
    base = baseline_exps(Wd, fmt)
    ue = base.astype(np.float64) + ( - 2.0 if KIND[fmt] == "mxint" else 0.0)
    bidx = np.repeat(np.arange(base.shape[1]), BLOCK)[:cols]
    unit = np.exp2(ue[:, bidx])
    t = np.abs(Wd) / unit
    om = g[None, :] * unit * unit
    k = np.minimum((t * NBINS_PER_UNIT).astype(np.int64), NB_T)
    if classes is None:
        classes = block_classes_fmt(t, bidx, rows, base.shape[1], ncls, Wd)
    cmat = classes[:, bidx]
    signsplit = KIND[fmt] != "mxfp"
    centers = (np.arange(NB_T + 1) + 0.5) / NBINS_PER_UNIT
    nc = int(classes.max()) + 1
    errs = np.zeros((nc, len(OFFSETS)))
    sgn = [(0, np.ones_like(Wd, bool))] if not signsplit else [(0, Wd >= 0), (1, Wd < 0)]
    for gi, smask in sgn:
        key = (cmat * (NB_T + 1) + k)[smask]; w0 = om[smask]; tt = t[smask]
        m0 = np.bincount(key, w0, nc * (NB_T + 1)).reshape(nc, - 1)
        m1 = np.bincount(key, w0 * tt, nc * (NB_T + 1)).reshape(nc, - 1)
        m2 = np.bincount(key, w0 * tt * tt, nc * (NB_T + 1)).reshape(nc, - 1)
        for oi, o in enumerate(OFFSETS):
            q = _code_round(centers / 2.0 ** o, fmt, neg = (gi == 1)) * 2.0 ** o
            errs[:, oi] += (m2 - 2 * q[None, :] * m1 + (q * q)[None, :] * m0).sum(axis = 1)
    if counters is not None:
        counters["hist_elems"] += Wd.size; counters["bin_reads"] += len(sgn) * (NB_T + 1) * len(OFFSETS) * nc
    j = errs.argmin(axis = 1)
    S = float(np.maximum(errs[np.arange(nc), j], 0).sum())
    return S, [OFFSETS[i] for i in j], errs, classes


## 4. Improve scales with local moves

Start from the histogram's block exponents. For each requested weight row, try nearby exponents for one block at a time, and also try nearby choices for pairs of blocks together. Measure the actual calibration-output error and accept improvements.

**Example:** suppose two blocks contribute output errors `+0.4` and `−0.3`. Their combined error is `0.1`. A scale change that makes the second error `−0.4` cancels them completely. Looking at the combined output matters, rather than minimizing each block's error separately.

The selector uses `hw = 3` by default, meaning local moves try offsets up to ±3 from the current choices, within the cached baseline range of ±6. Previously calculated candidates are reused. Stop when a sweep makes no improvement or the sweep limit is reached.

Pair moves test combinations for two blocks at a time. They do not enumerate every combination across all blocks, so this is a local search and can miss a better overall solution.



In [5]:
HALF = 6

NOFF = 2 * HALF + 1

def pair_refine(Wd, F, fmt, start_off, row_idx = None, max_sweeps = 10, hw = 2):
    """Coordinate / pair-move descent on the true calibration error: sweep over all single blocks and all block PAIRS of a row,
    each time trying every exponent combination within +/-hw of the current exponents and keeping the best.  The error never increases."""
    rows_all, cols = Wd.shape
    rows = np.arange(rows_all) if row_idx is None else np.asarray(row_idx, dtype = np.int64)
    R = len(rows); rng = block_ranges(cols); nb = len(rng); k = F.shape[1]
    base = baseline_exps(Wd, fmt)[rows]
    cur = np.clip(np.asarray(start_off)[rows] + HALF, hw, NOFF - 1 - hw).astype(np.int64)
    Dc = np.zeros((R, nb, NOFF, k)); Qc = [np.zeros((R, NOFF, e - s)) for s, e in rng]
    filled = np.zeros((R, nb, NOFF), dtype = bool); stats = {"wq": 0, "macs": 0, "sweeps": 0}
    deltas = np.array([0] + [s_ * m for m in range(1, hw + 1) for s_ in ( - 1, 1)]); M = len(deltas)
    ar = np.arange(R)

    def need(b, ib):
        s, e = rng[b]
        miss = ~np.take_along_axis(filled[:, b, :], ib, axis = 1)
        if not miss.any(): return
        rr, mm = np.nonzero(miss); oo = ib[rr, mm]
        Wb = Wd[rows[rr], s:e]; ex = base[rr, b] + (oo - HALF)
        Q = quantize(Wb, fmt, ex[:, None])
        Qc[b][rr, oo] = Q; Dc[rr, b, oo] = (Wb - Q) @ F[s:e]; filled[rr, b, oo] = True
        stats["wq"] += Q.size; stats["macs"] += Q.shape[0] * Q.shape[1] * k
    for b in range(nb): need(b, cur[:, b:b + 1])
    Dsum = sum(Dc[ar, b, cur[:, b]] for b in range(nb)); err = np.einsum("rk,rk->r", Dsum, Dsum)
    pairs = [(b, ) for b in range(nb)] + [(b, c) for b in range(nb) for c in range(b + 1, nb)]
    for sw in range(max_sweeps):
        changed = False
        for pr in pairs:
            cands = [np.clip(cur[:, b][:, None] + deltas[None, :], 0, NOFF - 1) for b in pr]
            for b, cb in zip(pr, cands): need(b, cb)
            rest = Dsum - sum(Dc[ar, b, cur[:, b]] for b in pr)
            if len(pr) == 1:
                tot = rest[:, None, :] + Dc[ar[:, None], pr[0], cands[0]]
                e = np.einsum("rmk,rmk->rm", tot, tot); am = e.argmin(1); ne = e[ar, am]
                newc = [cands[0][ar, am]]
            else:
                tot = rest[:, None, None, :] + Dc[ar[:, None], pr[0], cands[0]][:, :, None, :] + Dc[ar[:, None], pr[1], cands[1]][:, None, :, :]
                e = np.einsum("rijk,rijk->rij", tot, tot).reshape(R, - 1); am = e.argmin(1); ne = e[ar, am]
                newc = [cands[0][ar, am // M], cands[1][ar, am % M]]
            stats["macs"] += e.size * k
            imp = ne < err * (1 - 1e-12)
            if imp.any():
                for b, nc in zip(pr, newc): cur[imp, b] = nc[imp]
                Dsum = np.where(imp[:, None], rest + sum(Dc[ar, b, cur[:, b]] for b in pr), Dsum); err = np.where(imp, ne, err); changed = True
        stats["sweeps"] = sw + 1
        if not changed: break
    Wq = np.zeros_like(Wd); Wqr = np.zeros((R, cols))
    for b, (s, e_) in enumerate(rng): Wqr[:, s:e_] = Qc[b][ar, cur[:, b]]
    Wq[rows] = Wqr; full = np.zeros(rows_all); full[rows] = err
    stats["sse_rows"] = full; stats["offsets"] = cur - HALF
    return Wq, stats


## 5. Run the selection stages

**Check inputs:** reject incompatible shapes or non-finite values. If all weights or inputs are zero, all formats have zero output error and the fixed format order chooses one.

**Try an early decision:** calculate histogram scores and input coherence, which measures how much input energy concentrates in one shared direction. If the score gap is large enough and coherence is low enough, choose the histogram leader, refine its scales on all rows, and return it. These are practical decision rules, not guarantees.

**Otherwise, sample rows:** rank weight rows by input-weighted weight energy. Include leading rows up to the configured energy target and row-count cap, then take evenly spaced positions from the remaining ranked rows. Refine these rows for every format. Scale the tail-sample error to estimate the rest of the tail.

**Keep plausible candidates:** compare each candidate's estimated disadvantage with variation in the sampled row differences. The `z` setting controls the tolerance. Candidates with identical checked-row errors are represented by the sample leader at this step; equality on sampled rows does not prove equality everywhere. By default, also keep the histogram leader.

**Finish survivors:** refine remaining rows for each survivor, add the full row errors, and choose the smallest total. Return its rounded weights. Eliminated formats are not fully evaluated, so the final choice is best among the surviving locally refined candidates, not necessarily among all possible format-and-scale choices.



In [6]:
"""Define quantization and bounded scale-search helpers."""

DEFAULTS = dict(hw = 3, n_tail = 15, head_frac = 0.5, head_cap = 0.25, z = 3.0, early_margin = 1.3, early_coh = 0.10, ncls = 8, keep_hist_leader = True)

# Use histogram screening, sampled row checks, then survivor verification.
def select_format(W, X_cal, ** kw):
    P = dict(DEFAULTS, ** kw)
    Wd = np.asarray(W, dtype = np.float64); X = np.asarray(X_cal, dtype = np.float64)
    if Wd.ndim != 2 or X.ndim != 2 or Wd.shape[1] != X.shape[0]: raise ValueError(f"incompatible shapes W{Wd.shape} X_cal{X.shape}")
    if not (np.isfinite(Wd).all() and np.isfinite(X).all()): raise ValueError("W and X_cal must be finite (no NaN/Inf)")
    rows = Wd.shape[0]
    info = {"stage": None, "wq_events": 0, "macs": 0, "hist_elems": 0, "formats_verified_full": 0}
    out = {"info": info}
    if not Wd.any() or not X.any():
        info["stage"] = "0-degenerate"; out.update(chosen = FORMATS[0], tied = list(FORMATS), Wq = np.zeros_like(Wd), errors = None); return out
    G = X @ X.T; g = np.diag(G).copy(); lam = np.linalg.eigvalsh(G); coh = float(lam[ - 1] / lam.sum()); F = gram_factor(X); out["coherence"] = coh
    cnt = {"hist_elems": 0, "bin_reads": 0}; S, start = {}, {}
    for f in FORMATS:
        s_, offs, _, cls = surrogate(Wd, g, f, cnt, ncls = P["ncls"]); S[f] = s_; start[f] = np.asarray(offs)[cls]
    info["hist_elems"] = cnt["hist_elems"]; out["surrogate"] = S
    order = sorted(FORMATS, key = lambda f: (S[f], FORMATS.index(f))); sb = S[order[0]]
    d = [S[f] / sb for f in order[1:] if S[f] > 1.001 * sb] if sb > 0 else []
    margin = d[0] if d else np.inf; out["margin"] = margin

    def refine(f, ridx):
        Wq, st = pair_refine(Wd, F, f, start[f], row_idx = ridx, hw = P['hw']); info["wq_events"] += st["wq"]; info["macs"] += st["macs"]; return Wq, st["sse_rows"]
    if sb > 0 and np.isfinite(margin) and margin >= P["early_margin"] and coh < P["early_coh"]:
        Wq, sse = refine(order[0], None); info.update(stage = "1-histogram-accept", formats_verified_full = 1)
        out.update(chosen = order[0], Wq = Wq, errors = {order[0]: 100 * np.sqrt(sse.sum()) / max(np.linalg.norm(Wd @ X), 1e-300)}, tied = [order[0]]); return out
    e = ((Wd * Wd) * g[None, :]).sum(1); ordr = np.argsort( - e, kind = "stable"); cs = np.cumsum(e[ordr]) / e.sum()
    nh = int(min(np.searchsorted(cs, P["head_frac"]) + 1, P["head_cap"] * rows)); head, tail = ordr[:nh], ordr[nh:]; nt = P["n_tail"]
    pos = np.linspace(0, len(tail) - 1, nt + 2)[1: - 1].round().astype(int) if len(tail) > nt else np.arange(len(tail))
    samp = tail[np.unique(pos)]; scale = len(tail) / max(len(samp), 1)
    used = np.sort(np.concatenate([head, samp])); rest = np.setdiff1d(np.arange(rows), used)
    Wq_f, sse_f = {}, {}
    for f in FORMATS: Wq_f[f], sse_f[f] = refine(f, used)
    A_ = {f: sse_f[f][head].sum() + scale * sse_f[f][samp].sum() for f in FORMATS}
    b = min(FORMATS, key = lambda f: (A_[f], FORMATS.index(f))); surv = [b]
    for f in FORMATS:
        if f == b: continue
        dd = sse_f[f][samp] - sse_f[b][samp]; dh = (sse_f[f][head] - sse_f[b][head]).sum()
        if not dd.any() and not dh: continue
        mu = dh + scale * dd.sum(); se = scale * dd.std(ddof = 1) * np.sqrt(len(samp)) if len(samp) > 1 else 0.0
        if mu <= P["z"] * se + 1e-12 * abs(A_[b]): surv.append(f)
    if P["keep_hist_leader"] and sb > 0 and order[0] not in surv: surv.append(order[0])
    T2 = {}
    for f in surv:
        if len(rest):
            Wq, sse = refine(f, rest); Wq_f[f][rest] = Wq[rest]; sse_f[f][rest] = sse[rest]
        T2[f] = sse_f[f].sum()
    chosen = min(surv, key = lambda f: T2[f])
    info.update(stage = "3-sampled-verification", formats_verified_full = len(surv), survivors = surv, sampled_rows = int(len(used)))
    out.update(chosen = chosen, Wq = Wq_f[chosen], tied = [f for f in surv if T2[f] <= T2[chosen] * (1 + 1e-9) + 1e-300],
               errors = {f: 100 * np.sqrt(T2[f]) / max(np.linalg.norm(Wd @ X), 1e-300) for f in surv})
    return out


## 6. Check rounding consistency

Compare the vectorized quantizer with separate scalar checking functions included here. Test block scales, rounded values, and exact rounding ties. This checks the rounding implementations, not the accuracy of format selection.



In [7]:
def nearest_grid_rne(values, grid):
    values = np.asarray(values, dtype = np.float64)
    out = np.empty_like(values)
    for i, x in enumerate(values):
        d = np.abs(grid - x)
        dmin = np.min(d)
        candidates = np.where(np.isclose(d, dmin, rtol = 0.0, atol = 1e-15))[0]
        if len(candidates) == 1:
            idx = int(candidates[0])
        else:
            even = candidates[candidates % 2 == 0]
            idx = int(even[0] if len(even) else candidates[0])
        out[i] = grid[idx]
    return out

def _bfp_gamma(block, E):
    """Gamma of the paper: max |mantissa| produced at scale kappa=2^E,
    saturated at 2^(N-1)-1 (the paper's overflow test is Gamma >= 2^(N-1)-1)."""
    N = FORMAT_CONFIGS["BFP4"]["mantissa_bits"]
    m = np.rint(np.abs(np.asarray(block, dtype = np.float64)) / np.exp2(float(E)))
    return float(min(np.max(m), 2 ** (N - 1) - 1))

def bfp_exp_autoflex_predict(block):
    """Paper Algorithm 2 (line 9-10) for a static tensor: one observation,
    std(f)=0, no previous kappa, alpha=BFP_ALPHA.   E = ceil(log2 chi) - N + 1."""
    N = FORMAT_CONFIGS["BFP4"]["mantissa_bits"]
    chi = BFP_ALPHA * float(np.max(np.abs(block)))
    if chi == 0:
        return 0
    return int(np.ceil(np.log2(chi))) - N + 1

def bfp_exp_autoflex_init(block):
    """Paper Algorithm 1 (trial-and-error initialisation), literal.
    kappa = 2^E starts at 1 (E=0). Overflow: kappa *= 2^floor((N-1)/2).
    Underflow (Gamma < 2^(N-2)): kappa *= 2^(ceil(log2 max(Gamma,1)) - (N-2)),
    accepted only if Gamma > 2^(floor((N-1)/2)-2) ('enough bits for a reliable jump')."""
    N = FORMAT_CONFIGS["BFP4"]["mantissa_bits"]
    if float(np.max(np.abs(block))) == 0:
        return 0
    E = 0
    for _ in range(1024):
        G = _bfp_gamma(block, E)
        if G >= 2 ** (N - 1) - 1:
            E += (N - 1) // 2
        elif G < 2 ** (N - 2):
            E += int(np.ceil(np.log2(max(G, 1.0)))) - (N - 2)
            if G > 2.0 ** ((N - 1) // 2 - 2):
                break
        else:
            break
        E = int(np.clip(E, E8M0_MIN_EXP, E8M0_MAX_EXP))
    return E

def bfp_shared_exponent(block):
    if BFP_EXP_POLICY == "autoflex_init":
        return bfp_exp_autoflex_init(block)
    if BFP_EXP_POLICY == "autoflex_predict":
        return bfp_exp_autoflex_predict(block)
    raise ValueError(f"Unknown BFP_EXP_POLICY={BFP_EXP_POLICY}")

def baseline_exp(block, fmt):
    amax = float(np.max(np.abs(block)))
    if amax == 0:
        return 0

    raw = int(np.floor(np.log2(amax)))
    cfg = FORMAT_CONFIGS[fmt]

    if cfg["kind"] == "mxfp":
        exp = raw + cfg["baseline_exp_shift"]
    elif cfg["kind"] == "mxuint":

        neg_extent = max(0.0, - float(np.min(block)))
        pos_extent = max(0.0, float(np.max(block)))
        required_step = max(neg_extent / 8.0, pos_extent / 7.0)
        exp = int(np.ceil(np.log2(required_step))) if required_step > 0 else 0
    elif cfg["kind"] == "bfp":

        exp = bfp_shared_exponent(block)
    else:
        exp = raw

    if "exp_min" in cfg:
        exp = int(np.clip(exp, cfg["exp_min"], cfg["exp_max"]))
    return exp

def quantize_block_at_exp(block, fmt, exp, return_events = False):
    block = np.asarray(block, dtype = np.float64)
    cfg = FORMAT_CONFIGS[fmt]

    if "exp_min" in cfg:
        exp = int(np.clip(exp, cfg["exp_min"], cfg["exp_max"]))

    zeroed = np.zeros(block.shape, dtype = bool)
    saturated = np.zeros(block.shape, dtype = bool)

    if cfg["kind"] == "mxfp":
        scale = np.exp2(exp)
        local = np.abs(block) / scale
        qlocal = nearest_grid_rne(local, cfg["grid"])
        qblock = np.sign(block) * qlocal * scale
        zeroed = (block != 0) & (qblock == 0)

        grid = cfg["grid"]
        if len(grid) >= 2:
            final_step = grid[ - 1] - grid[ - 2]
            sat_threshold = grid[ - 1] + 0.5 * final_step
            saturated = local > sat_threshold

    elif cfg["kind"] == "mxint":
        step = np.exp2(exp) * cfg["element_scale"]
        q_raw = np.rint(block / step)
        q = np.clip(q_raw, cfg["qmin"], cfg["qmax"])
        qblock = q * step
        zeroed = (block != 0) & (q == 0)
        saturated = (q_raw < cfg["qmin"]) | (q_raw > cfg["qmax"])

    elif cfg["kind"] == "bfp":

        scale = np.exp2(exp)
        m_raw = np.rint(block / scale)
        m = np.clip(m_raw, cfg["qmin"], cfg["qmax"])
        qblock = m * scale
        zeroed = (block != 0) & (m == 0)
        saturated = (m_raw < cfg["qmin"]) | (m_raw > cfg["qmax"])

    elif cfg["kind"] == "mxuint":

        scale = np.exp2(exp)
        q_raw = np.rint(block / scale) + 8
        q = np.clip(q_raw, 0, 15)
        q_signed = q - 8
        qblock = q_signed * scale
        zeroed = (block != 0) & (q_signed == 0)
        saturated = (q_raw < 0) | (q_raw > 15)

    else:
        raise ValueError(f"Numerical quantizer not defined for {fmt}")

    if return_events:
        return qblock, zeroed, saturated
    return qblock

def _selftest(n_blocks = 300, seed = 0):
    rng = np.random.default_rng(seed); bad = 0; total = 0
    for fmt in FORMATS:
        for t in range(n_blocks):
            kind = t % 6
            if kind == 0: blk = rng.normal(size = 32)
            elif kind == 1: blk = rng.normal(size = 32) * 10.0 ** rng.uniform( - 3, 3)
            elif kind == 2: blk = np.full(32, rng.normal())
            elif kind == 3: blk = np.exp2(rng.integers( - 6, 6, 32)) * rng.choice([ - 1, 1], 32)
            elif kind == 4:
                blk = rng.choice([.25, .75, 1.25, 1.75, 2.5, 3.5, 5.0], 32) * np.exp2(rng.integers( - 3, 3)) * rng.choice([ - 1, 1], 32)
            else: blk = - np.abs(rng.normal(size = 32))
            e_ref = baseline_exp(blk, fmt); e_vec = int(baseline_exps(blk[None, :], fmt)[0, 0])
            for off in ( - 2, 0, 1):
                q_ref = quantize_block_at_exp(blk, fmt, e_ref + off)
                q_vec = quantize(blk[None, :], fmt, np.array([[e_ref + off]]))[0]
                total += 1; bad += int(e_ref != e_vec or not np.array_equal(q_ref, q_vec))
    return bad, total
bad, total = _selftest()
print(f"convention self-test: {total - bad}/{total} block quantisations identical to the brute-force notebook's scalar code")
assert bad == 0


convention self-test: 4500/4500 block quantisations identical to the brute-force notebook's scalar code


## 7. Check that selector functions avoid exhaustive search

Inspect the listed selector functions and their nested compiled code for forbidden reference-search names. The assertion fails if it finds one. This is a static check of those named functions, rather than a general proof about every possible execution path.



In [8]:
import types
ALGO_FUNCS = [select_format, pair_refine, surrogate, block_classes_fmt, quantize, baseline_exps, gram_factor, block_ranges, _fp_round, _code_round]
FORBIDDEN = {"exact_search", "_solve", "offsets_for", "strong_reference_search"}

def _names(fn):
    out = set(fn.__code__.co_names)
    for c in fn.__code__.co_consts:
        if isinstance(c, types.CodeType): out |= set(c.co_names)
    return out
for fn in ALGO_FUNCS:
    hit = _names(fn) & FORBIDDEN
    assert not hit, f"{fn.__name__} references {hit}"
print("OK: none of", [f.__name__ for f in ALGO_FUNCS], "references", sorted(FORBIDDEN))


OK: none of ['select_format', 'pair_refine', 'surrogate', 'block_classes_fmt', 'quantize', 'baseline_exps', 'gram_factor', 'block_ranges', '_fp_round', '_code_round'] references ['_solve', 'exact_search', 'offsets_for', 'strong_reference_search']


## 8. Exhaustive reference for evaluation only

This separate reference tries all block-exponent combinations within its candidate window. It starts within ±3 of baseline exponents and can widen up to ±5 when the best choice touches a boundary.

It minimizes complete calibration-output error within the searched candidates. It does not search every legal exponent. The selector does not call this reference; later cells use it to evaluate the selector's choice.



In [9]:
def offsets_for(window):
    return sorted(range( - window, window + 1), key = lambda o: (abs(o), o))

def _solve(Wd, F, fmt, base, rows, w, stats):
    """exact enumeration for `rows` at window w. returns (idx per block, sse, Wq rows, offs per block)."""
    rngs = block_ranges(Wd.shape[1]); nb = len(rngs)
    offs = np.array(offsets_for(w)); K = len(offs)
    Dl, Qs = [], []
    for b, (s, e) in enumerate(rngs):
        Wb = Wd[rows, s:e]
        ex = base[rows, b][:, None] + offs[None, :]
        Q = quantize(Wb[:, None, :], fmt, ex[:, :, None])
        stats["wq"] += Q.size
        Qs.append(Q)
        Dl.append((Wb[:, None, :] - Q) @ F[s:e])
    h = (nb + 1) // 2

    def comb(lst):
        S = lst[0]
        for d in lst[1:]:
            S = (S[:, :, None, :] + d[:, None, :, :]).reshape(S.shape[0], - 1, S.shape[ - 1])
        return S
    L, Rr = comb(Dl[:h]), comb(Dl[h:])
    nl = np.einsum("rik,rik->ri", L, L); nr = np.einsum("rik,rik->ri", Rr, Rr)
    tot = nl[:, :, None] + nr[:, None, :] + 2.0 * np.einsum("rik,rjk->rij", L, Rr)
    flat = tot.reshape(len(rows), - 1)

    M = min(48, flat.shape[1])
    cand = np.argpartition(flat, M - 1, axis = 1)[:, :M] if M < flat.shape[1] else np.tile(np.arange(flat.shape[1]), (len(rows), 1))
    ci, cj = cand // Rr.shape[1], cand % Rr.shape[1]
    ar = np.arange(len(rows))[:, None]
    dv = L[ar, ci] + Rr[ar, cj]
    ex = np.einsum("rmk,rmk->rm", dv, dv)
    mn = ex.min(axis = 1)
    tied = ex <= mn[:, None] * (1 + 1e-12)
    am = np.where(tied, cand, np.iinfo(np.int64).max).min(axis = 1)

    noise = 4e-15 * (nl.max(axis = 1) + nr.max(axis = 1))
    fs = np.partition(flat, M - 1, axis = 1)[:, M - 1] if M < flat.shape[1] else flat.max(axis = 1)
    amb = (fs - flat.min(axis = 1)) <= noise
    for r in np.nonzero(amb)[0]:
        tot_r = L[r][:, None, :] + Rr[r][None, :, :]
        e_r = np.einsum("ijk,ijk->ij", tot_r, tot_r).ravel()
        k0 = int(np.argmin(e_r)); am[r] = k0; mn[r] = e_r[k0]
        stats["wq"] += 0
    i, j = am // Rr.shape[1], am % Rr.shape[1]
    sse = mn
    kl = [K] * h; kr = [K] * (nb - h)
    idx = np.zeros((len(rows), nb), dtype = np.int64)
    for side, ii, ks, off in ((0, i, kl, 0), (1, j, kr, h)):
        rem = ii.copy()
        for pos in range(len(ks) - 1, - 1, - 1):
            idx[:, off + pos] = rem % ks[pos]; rem //= ks[pos]
    Wq = np.zeros((len(rows), Wd.shape[1]))
    for b, (s, e) in enumerate(rngs):
        Wq[:, s:e] = Qs[b][np.arange(len(rows)), idx[:, b]]
    return idx, sse, Wq, offs[idx]

# Search bounded exponent combinations for the requested rows.
def exact_search(Wd, F, fmt, W0 = 3, WMAX = 5, row_idx = None):
    """Same search as strong_reference_search: exhaustive over +/-W0 exponents around baseline per block,
    widened row-wise (up to WMAX) while the optimum sits on the window edge."""
    rows = Wd.shape[0]
    base = baseline_exps(Wd, fmt)
    Wq = np.zeros_like(Wd); offs_out = np.zeros(base.shape, dtype = np.int64); sse_out = np.zeros(rows)
    stats = {"wq": 0}
    pending = np.arange(rows) if row_idx is None else np.asarray(row_idx, dtype = np.int64); w = W0
    while len(pending):
        idx, sse, Wqr, offs = _solve(Wd, F, fmt, base, pending, w, stats)
        edge = (np.abs(offs) == w).any(axis = 1)
        done = ~(edge & (w < WMAX))
        Wq[pending[done]] = Wqr[done]; offs_out[pending[done]] = offs[done]; sse_out[pending[done]] = sse[done]
        pending = pending[~done]; w += 1
    stats['sse_rows'] = sse_out
    return Wq, offs_out, stats


## 9. Run on the configured data

Load the weights and calibration inputs, run the selector, and display its choice, decision stage, error, and work counts.



In [10]:
W_in = np.load(W_PATH); X_in = np.load(X_CAL_PATH)
X_test_in = np.load(X_TEST_PATH) if X_TEST_PATH else None
Wd = W_in.astype(np.float64); Xd = X_in.astype(np.float64)
print("W", W_in.shape, W_in.dtype, "| X_cal", X_in.shape, "| negative weights: %.2f%%" % (100 * np.mean(Wd < 0)), "| zero weights: %.2f%%" % (100 * np.mean(Wd == 0)))
t0 = time.perf_counter(); result = select_format(W_in, X_in, ** GHS_PARAMS); t_ghs = time.perf_counter() - t0
info = result["info"]
print("\n=== GHS-2 RESULT ===")
print("Selected format :", result["chosen"], "| tied best:", result["tied"])
print("Decided at stage:", info["stage"])
print("Histogram surrogate (diag-input model, lower = better):", {k: round(v, 4) for k, v in result.get("surrogate", {}).items()})
print("Histogram margin: %s | input coherence (top eigen-share of XXt): %s" % (result.get("margin"), result.get("coherence")))
print("Weight-quantisation events: %d | multiply-accumulates: %d" % (info["wq_events"], info["macs"]))
print("Formats refined on all rows:", info.get("formats_verified_full"), "| rows in the sampling stage:", info.get("sampled_rows"))
Wq = result["Wq"]
print("Output error of the returned matrix: %.6f %%" % err_pct(Wd, Wq, Xd))
if X_test_in is not None: print("Held-out X_test error            : %.6f %%" % err_pct(Wd, Wq, X_test_in.astype(np.float64)))
else: print("X_test not supplied -> held-out evaluation skipped.")
print("selector time: %.3f s" % t_ghs)


W (100, 100) float32 | X_cal (100, 256) | negative weights: 49.48% | zero weights: 0.00%

=== GHS-2 RESULT ===
Selected format : E2M1 | tied best: ['E2M1']
Decided at stage: 3-sampled-verification
Histogram surrogate (diag-input model, lower = better): {'E2M1': 372.4651, 'E1M2': 514.426, 'MXUINT4': 590.5072, 'MXINT4': 481.4204, 'BFP4': 481.4204}
Histogram margin: 1.292524979687217 | input coherence (top eigen-share of XXt): 0.025380551333585487
Weight-quantisation events: 185264 | multiply-accumulates: 35270400
Formats refined on all rows: 1 | rows in the sampling stage: 40
Output error of the returned matrix: 12.030660 %
X_test not supplied -> held-out evaluation skipped.
selector time: 0.189 s


## 10. Compare with the full reference

Evaluate all formats using the exhaustive reference, then compare with the selector's returned matrix. Lower full-output error is better. This reference work evaluates selection quality and is separate from the selector's own work.



In [11]:
if RUN_REFERENCE_CHECK:
    F_ = gram_factor(Xd); rows_ = []; t0 = time.perf_counter()
    for f in FORMATS:
        Wq_f, offs_f, st_f = exact_search(Wd, F_, f)
        rows_.append(dict(Format = f, Reference_Error_pct = err_pct(Wd, Wq_f, Xd), Weight_quantisations = st_f["wq"]))
    ref = pd.DataFrame(rows_).sort_values("Reference_Error_pct").reset_index(drop = True); t_ref = time.perf_counter() - t0
    best = ref.Reference_Error_pct.min(); got = err_pct(Wd, result["Wq"], Xd)
    display(ref)
    print("Reference best error: %.6f %% | GHS-2 returned-matrix error: %.6f %% | extra: %.4f %%" % (best, got, 100 * (got - best) / max(best, 1e-12)))
    print("Selected format is among the reference-best (ties accepted):", float(ref.loc[ref.Format == result["chosen"], "Reference_Error_pct"].iloc[0]) <= best * (1 + 1e-9))
    print("Quantisations: GHS-2 %d vs exhaustive %d (%.1f %%) | time %.3f s vs %.3f s" % (info["wq_events"], ref.Weight_quantisations.sum(), 100 * info["wq_events"] / ref.Weight_quantisations.sum(), t_ghs, t_ref))


,Format,Reference_Error_pct,Weight_quantisations
0,E2M1,12.030660,71800
1,MXUINT4,13.452060,70000
2,MXINT4,13.452060,70000
3,BFP4,13.452060,70000
4,E1M2,14.004212,70000


Reference best error: 12.030660 % | GHS-2 returned-matrix error: 12.030660 % | extra: 0.0000 %
Selected format is among the reference-best (ties accepted): True
Quantisations: GHS-2 185264 vs exhaustive 351800 (52.7 %) | time 0.189 s vs 0.210 s


## 11. Optional larger tests

Enable the test-suite setting to run generated and edge cases. This section needs the separate importable modules `generate_1920_cases.py` and `generate_edge_cases.py`; they are not included here.

Choose formats and tune scales using calibration inputs. Evaluate the returned matrices on independent test inputs afterward and compare with the best test error among the calibration-tuned reference matrices. The CSV reports choices, extra error, and work counts. These optional tests are separate from the configured-data run.



In [12]:
def run_suite(max_cases = None):
    gens = []
    try: import generate_1920_cases as g1; gens.append(g1.iter_generated_suite())
    except Exception as e: print("generate_1920_cases not found:", e)
    try: import generate_edge_cases as g2; gens.append(g2.iter_edge_suite())
    except Exception as e: print("generate_edge_cases not found:", e)
    rm = lambda A, B, X: float(np.sqrt(np.mean(((A - B) @ X) ** 2)))
    out = []
    for c in itertools.chain( * gens):
        if max_cases and len(out) >= max_cases: break
        Wc, Xc, Xt = c["W"].astype(np.float64), c["X_cal"].astype(np.float64), c["X_test"].astype(np.float64)
        r = select_format(Wc, Xc, ** GHS_PARAMS); Fc = gram_factor(Xc); T, Tt = {}, {}
        if Wc.any() and Xc.any():
            for f in FORMATS:
                Wf, _, st = exact_search(Wc, Fc, f); T[f] = rm(Wc, Wf, Xc); Tt[f] = rm(Wc, Wf, Xt)
        else: T = {f: 0.0 for f in FORMATS}; Tt = dict(T)
        ch = r["chosen"]; sc, st_ = rm(Wc, r["Wq"], Xc), rm(Wc, r["Wq"], Xt); bc, bt = min(T.values()), min(Tt.values())
        out.append(dict(case_id = c["case_id"], weight_shape = c["weight_shape"], input_pattern = c["input_pattern"], chosen = ch, stage = r["info"]["stage"],
                        cal_extra_pct = 100 * (sc - bc) / max(bc, 1e-6), test_extra_pct = 100 * (st_ - bt) / max(bt, 1e-6),
                        format_best_on_cal = T[ch] <= bc * (1 + 1e-9) + 1e-12, wq_events = r["info"]["wq_events"]))
    return pd.DataFrame(out)

if RUN_TEST_SUITE:
    suite = run_suite(); suite.to_csv("ghs2_suite_results.csv", index = False)
    for name, d in (("generated", suite[~suite.case_id.str.startswith("edge_")]), ("edge", suite[suite.case_id.str.startswith("edge_")])):
        print(f"{name}: n={len(d)} | within 1% (test) {int((d.test_extra_pct <= 1).sum())} | within 5% (test) {int((d.test_extra_pct <= 5).sum())} | worst test {d.test_extra_pct.max():.2f}% | "
              f"within 1% (cal) {int((d.cal_extra_pct <= 1).sum())} | within 5% (cal) {int((d.cal_extra_pct <= 5).sum())} | format best on cal {int(d.format_best_on_cal.sum())}")
    print(suite.groupby("input_pattern").apply(lambda d: pd.Series(dict(n = len(d), w1 = int((d.test_extra_pct <= 1).sum()), w5 = int((d.test_extra_pct <= 5).sum()), worst = round(d.test_extra_pct.max(), 2)))))
else: print("RUN_TEST_SUITE is False — skipped.")


RUN_TEST_SUITE is False — skipped.
